# AdaLoRA-Lite Train And Paper-Aligned Eval

Train the dynamic AdaLoRA-Lite extension on E2E with GPT-2 Medium, back up the training run to Drive immediately after training, then run paper-aligned official-beam generation/evaluation and back up the evaluated artifacts again.

Primary comparison target: `late_ramp_r4` from the existing paper-aligned official beam results.

## 1. GPU And Repo Setup

Use a GPU runtime. This follows the existing project notebook pattern: mount Drive, clone or pull GitHub, install requirements, and work from `lora-gpt2-medium-e2e`.

In [ ]:
!nvidia-smi

import torch
print("cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0))

In [ ]:
from getpass import getpass
from pathlib import Path
import json
import os
import re
import shutil
import subprocess

from google.colab import drive
import pandas as pd
import yaml

drive.mount('/content/drive')

REPO_OWNER = 'justinlxiang'
REPO_NAME = 'CS4782-final-project'
BRANCH = 'main'
PROJECT_DIR = Path('/content') / REPO_NAME
WORK_DIR = PROJECT_DIR / 'lora-gpt2-medium-e2e'

token = getpass('GitHub token, or press Enter for public clone: ')
repo_url = f'https://github.com/{REPO_OWNER}/{REPO_NAME}.git'
if token:
    repo_url = f'https://{token}@github.com/{REPO_OWNER}/{REPO_NAME}.git'

if PROJECT_DIR.exists():
    subprocess.run(['git', '-C', str(PROJECT_DIR), 'fetch', 'origin'], check=True)
    subprocess.run(['git', '-C', str(PROJECT_DIR), 'checkout', BRANCH], check=True)
    subprocess.run(['git', '-C', str(PROJECT_DIR), 'pull', '--ff-only', 'origin', BRANCH], check=True)
else:
    subprocess.run(['git', 'clone', '--branch', BRANCH, repo_url, str(PROJECT_DIR)], check=True)

os.chdir(WORK_DIR)
print('working directory:', Path.cwd())
!git log --oneline -3
!pip install -q -r requirements.txt

## 2. Data Prep

Download the same E2E files used by the existing notebooks and prepare processed JSONL/reference files.

In [ ]:
!mkdir -p data/raw/e2e
!curl -L -o data/raw/e2e/train.txt https://raw.githubusercontent.com/microsoft/LoRA/main/examples/NLG/data/e2e/train.txt
!curl -L -o data/raw/e2e/valid.txt https://raw.githubusercontent.com/microsoft/LoRA/main/examples/NLG/data/e2e/valid.txt
!curl -L -o data/raw/e2e/test.txt https://raw.githubusercontent.com/microsoft/LoRA/main/examples/NLG/data/e2e/test.txt
!python scripts/prepare_e2e.py --config configs/e2e_gpt2_medium_adalora_lite.yaml
!wc -l data/raw/e2e/*.txt data/processed/e2e_gpt2/*.jsonl

## 3. AdaLoRA-Lite Config And Paths

The config uses `r_max=8`, target active rank budget `192`, delayed pruning, and paper-aligned official beam decoding.

In [ ]:
CONFIG_PATH = Path('configs/e2e_gpt2_medium_adalora_lite.yaml')
RUN_DIR = Path('outputs/runs/adalora_lite_rmax8_target192_late_prune')
DRIVE_RUN_DIR = Path('/content/drive/MyDrive/lora_adalora_lite_rmax8_target192_late_prune')
ADAPTER_PATH = RUN_DIR / 'checkpoints' / 'adapter_final.pt'

cfg = yaml.safe_load(CONFIG_PATH.read_text())
cfg['generation']['decoder'] = 'official_beam'
cfg['generation']['num_beams'] = 10
cfg['generation']['length_penalty'] = 0.9
cfg['generation']['batch_size'] = 4
cfg['evaluation']['predictions_file'] = str(RUN_DIR / 'generations_test.txt')
cfg['evaluation']['references_file'] = 'data/processed/e2e_gpt2/references_test.txt'
cfg['project']['output_dir'] = str(RUN_DIR)
CONFIG_PATH.write_text(yaml.safe_dump(cfg, sort_keys=False))

print(CONFIG_PATH)
print('run dir:', RUN_DIR)
print('drive dir:', DRIVE_RUN_DIR)
print('adalora_lite:', cfg['lora']['adalora_lite'])

## 4. Train And Immediately Back Up To Drive

This cell saves the adapter/checkpoints to Drive right after training, before slower official-beam generation starts.

In [ ]:
!python scripts/count_params.py --config configs/e2e_gpt2_medium_adalora_lite.yaml
!python scripts/train.py --config configs/e2e_gpt2_medium_adalora_lite.yaml --train --device cuda

def copy_run_to_drive(stage):
    DRIVE_RUN_DIR.mkdir(parents=True, exist_ok=True)
    if DRIVE_RUN_DIR.exists():
        shutil.rmtree(DRIVE_RUN_DIR)
    shutil.copytree(RUN_DIR, DRIVE_RUN_DIR)
    shutil.copy2(CONFIG_PATH, DRIVE_RUN_DIR / CONFIG_PATH.name)
    (DRIVE_RUN_DIR / 'backup_stage.txt').write_text(stage)
    print(f'copied {stage} artifacts to {DRIVE_RUN_DIR}')

copy_run_to_drive('after_training')
!du -sh /content/drive/MyDrive/lora_adalora_lite_rmax8_target192_late_prune
!find /content/drive/MyDrive/lora_adalora_lite_rmax8_target192_late_prune -maxdepth 3 -type f | sort | tail -60

## 5. Official-Beam Generation And Evaluation

Generate with the trained adapter, run project metrics, then run the official E2E scorer.

In [ ]:
if not Path('external/e2e-metrics/.git').exists():
    subprocess.run(['git', 'clone', 'https://github.com/tuetschek/e2e-metrics.git', 'external/e2e-metrics'], check=True)

!python scripts/generate.py --config configs/e2e_gpt2_medium_adalora_lite.yaml --split test --adapter "$ADAPTER_PATH"
!python scripts/evaluate.py --config configs/e2e_gpt2_medium_adalora_lite.yaml --official

ref_file = RUN_DIR / 'generations_test.e2e_refs.txt'
pred_file = RUN_DIR / 'generations_test.e2e_preds.txt'
official_out = RUN_DIR / 'generations_test.official_e2e_metrics.txt'
if ref_file.exists() and pred_file.exists():
    with official_out.open('w') as handle:
        subprocess.run(
            ['python', 'external/e2e-metrics/measure_scores.py', str(ref_file), str(pred_file), '-p'],
            stdout=handle,
            stderr=subprocess.STDOUT,
            check=True,
        )
    print(official_out.read_text())

## 6. Summarize Allocation And Back Up Evaluated Artifacts

In [ ]:
allocation_path = RUN_DIR / 'adalora_lite_allocation.jsonl'
if allocation_path.exists():
    allocation_records = [json.loads(line) for line in allocation_path.read_text().splitlines() if line.strip()]
    final_allocation = allocation_records[-1]
    rows = final_allocation['layers']
    display(pd.DataFrame(rows)[['layer', 'query', 'key', 'value', 'total']])
    print('active rank units:', final_allocation['active_rank_units'])

metrics_path = RUN_DIR / 'generations_test.metrics.json'
summary = {'run': 'adalora_lite_rmax8_target192_late_prune'}
if metrics_path.exists():
    summary.update(json.loads(metrics_path.read_text()))
if official_out.exists():
    text = official_out.read_text()
    for key, pattern in {
        'official_bleu': r'BLEU: ([0-9.]+)',
        'official_meteor': r'METEOR: ([0-9.]+)',
        'official_rouge_l': r'ROUGE_L: ([0-9.]+)',
    }.items():
        match = re.search(pattern, text)
        if match:
            summary[key] = float(match.group(1))
summary_path = RUN_DIR / 'adalora_lite_summary.json'
summary_path.write_text(json.dumps(summary, indent=2, sort_keys=True))
print(summary_path.read_text())

copy_run_to_drive('after_official_beam_eval')
!du -sh /content/drive/MyDrive/lora_adalora_lite_rmax8_target192_late_prune
!find /content/drive/MyDrive/lora_adalora_lite_rmax8_target192_late_prune -maxdepth 3 -type f | sort | tail -100

In [ ]:
# Optional: export the learned masked adapter into a compact static LoRA checkpoint.
# This folds each active s_i value into B_i and writes a normal rank_pattern config.
STATIC_ADAPTER_PATH = RUN_DIR / 'checkpoints' / 'adapter_final_static_lora.pt'
STATIC_CONFIG_PATH = RUN_DIR / 'e2e_gpt2_medium_adalora_lite_static_config.json'
STATIC_ALLOCATION_PATH = RUN_DIR / 'adalora_lite_static_allocation.json'

!python scripts/export_adalora_lite_static.py \
  --config configs/e2e_gpt2_medium_adalora_lite.yaml \
  --adapter "$ADAPTER_PATH" \
  --output-adapter "$STATIC_ADAPTER_PATH" \
  --output-config "$STATIC_CONFIG_PATH" \
  --allocation-json "$STATIC_ALLOCATION_PATH"

print('static adapter:', STATIC_ADAPTER_PATH)
print('static config:', STATIC_CONFIG_PATH)
print('static allocation:', STATIC_ALLOCATION_PATH)
print(STATIC_ALLOCATION_PATH.read_text()[:4000])

# Preserve the compact export beside the already evaluated masked AdaLoRA-Lite run.
copy_run_to_drive('after_static_export')
!find /content/drive/MyDrive/lora_adalora_lite_rmax8_target192_late_prune -maxdepth 3 -type f | sort | tail -120